# LightGBM — 10-class & 4-class 비교

Phase 9 실험: Gradient Boosting Tree (LightGBM)을 두 설정으로 평가.

- 1차 발표: LightGBM 4-class 58.7%
- Phase 9 (3시즌 데이터): 10-class + 4-class 재평가

작성일: 2026-05-23

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

OUTPUT_DIR = Path('../outputs')

data_10 = np.load(OUTPUT_DIR / 'evaluation_lgb_10cls.npz', allow_pickle=True)
data_4  = np.load(OUTPUT_DIR / 'evaluation_lgb_4cls.npz',  allow_pickle=True)

cls10 = ['Ball','Strike','Single','Double','Triple','HomeRun','FieldOut','Strikeout','Walk','HitByPitch']
cls4  = ['Ball','Strike','Foul','InPlay']

print('=== 10-class ===')
print(f'Top-1: {float(data_10["top1"])*100:.1f}%  Top-3: {float(data_10["top3"])*100:.1f}%  CE: {float(data_10["ce"]):.4f}')
print(f'Best iteration: {int(data_10["best_iteration"])}')
print('=== 4-class ===')
print(f'Top-1: {float(data_4["top1"])*100:.1f}%  Top-3: {float(data_4["top3"])*100:.1f}%  CE: {float(data_4["ce"]):.4f}')
print(f'Best iteration: {int(data_4["best_iteration"])}')


## 1. 핵심 지표 요약

| 구분 | Top-1 | Top-3 | CE | Best iter |
|------|-------|-------|----|-----------|
| **10-class** | 41.1% | 86.2% | 1.4607 | 1 |
| **4-class** | 60.8% | 96.6% | 0.8729 | 500 |

### 10-class: Best iteration = 1
LightGBM이 첫 번째 트리 이후 **"No further splits with positive gain"** 경고와 함께
early stopping. 사실상 initial score (클래스 빈도 prior)만 사용한 trivial 모델.

Gradient boosting도 77-dim 단일 pitch feature만으로는 10-class 불균형 분류에서
개선 가능한 feature split을 발견하지 못함.

### 4-class: Best iteration = 500 (max rounds)
계속 개선 가능성 있음 (early stopping 미발동). 60.8% 달성.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

acc10 = data_10['per_class_accuracy']
colors10 = ['#e74c3c' if a < 0.01 else '#f39c12' if a < 0.5 else '#27ae60' for a in acc10]
axes[0].bar(cls10, acc10, color=colors10, alpha=0.8)
axes[0].set_title('LightGBM 10-class: Per-class Accuracy', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1.15)
axes[0].tick_params(axis='x', rotation=40)
for i, v in enumerate(acc10):
    axes[0].text(i, v + 0.02, f'{v:.0%}', ha='center', fontsize=8)

acc4 = data_4['per_class_accuracy']
colors4 = ['#e67e22'] * 4
axes[1].bar(cls4, acc4, color=colors4, alpha=0.8)
axes[1].set_title('LightGBM 4-class: Per-class Accuracy', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.15)
for i, v in enumerate(acc4):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrix — 4-class only (10-class is trivial)
cm = data_4['confusion_matrix']
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=cls4, yticklabels=cls4, ax=ax)
ax.set_title('LightGBM 4-class: Confusion Matrix', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
plt.tight_layout()
plt.show()


## 2. 1차 발표 비교

| | 1차 발표 | Phase 9 (3시즌) |
|--|---------|----------------|
| **데이터** | 2시즌 | 3시즌 (2022 추가) |
| **4-class Top-1** | 58.7% | **60.8%** (+2.1pp) |
| **4-class CE** | — | 0.8729 |

3시즌 데이터로 2.1pp 향상. 데이터 증가의 효과.

## 3. 분석

- **10-class**: Gradient boosting도 10-class 불균형에서는 trivial 모델로 collapse
- **4-class**: LightGBM의 tree ensemble이 효과적 — 60.8% (MLP 60.9%와 거의 동일)
- 단일 pitch feature 77-dim에서 tree와 neural net의 성능이 수렴하는 것은
  해당 feature set의 표현력 한계에 도달했음을 시사
- **시퀀스 정보 없이** 4-class ~61%가 단일 pitch 모델의 상한선으로 추정